# Flight Delay Prediction — Data Pipeline

This notebook covers the full data pipeline:
1. BTS flight data download
2. NOAA weather download and join
3. Holiday and weekend flags
4. Temporal train/val/test split

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import time
import os
import gc
import shutil
import glob
import requests
from io import StringIO, BytesIO
from zipfile import ZipFile
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import timedelta
import holidays
import pyarrow.parquet as pq
import pyarrow as pa

# For Jupyter notebooks — render plots inline
%matplotlib inline

print('✅ All imports successful!')

✅ All imports successful!


In [2]:
# Set data directory relative to project root
cwd = os.getcwd()
if os.path.basename(cwd) == 'notebooks':
    DRIVE_PATH = os.path.abspath(os.path.join(cwd, '..', 'data'))
else:
    DRIVE_PATH = os.path.abspath(os.path.join(cwd, 'data'))

os.makedirs(DRIVE_PATH, exist_ok=True)
print(f'Data directory: {DRIVE_PATH}')

## Section 1: BTS Flight Data Download

Downloads BTS On-Time Performance data (2018–2024) and saves as parquet.

In [ ]:
def load_bts_month(year, month):
    url = (
        f'https://transtats.bts.gov/PREZIP/'
        f'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip'
    )
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    z = ZipFile(BytesIO(response.content))
    csv_name = [f for f in z.namelist() if f.endswith('.csv')][0]
    cols = [
        'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
        'Reporting_Airline', 'Flight_Number_Reporting_Airline',
        'Origin', 'Dest', 'CRSDepTime', 'DepTimeBlk',
        'CRSArrTime', 'ArrDel15', 'CRSElapsedTime', 'Distance', 'DistanceGroup'
    ]
    df = pd.read_csv(z.open(csv_name), low_memory=False, usecols=cols)
    df['Year'] = year
    df['Month'] = month
    return df

tasks = [(y, m) for y in range(2018, 2025) for m in range(1, 13)]
frames, failed = [], []

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(load_bts_month, y, m): (y, m) for y, m in tasks}
    for future in as_completed(futures):
        y, m = futures[future]
        try:
            chunk = future.result()
            frames.append(chunk)
            print(f'✅ {y}-{m:02d}: {len(chunk):,} rows')
        except Exception as e:
            print(f'⚠️  {y}-{m:02d}: Skipped ({e})')
            failed.append((y, m))

bts = pd.concat(frames, ignore_index=True).sort_values('FlightDate').reset_index(drop=True)
print(f'\n🎉 Total: {len(bts):,} rows × {bts.shape[1]} columns')
print(f'📅 Range: {bts["FlightDate"].min()} → {bts["FlightDate"].max()}')

bts.to_parquet(f'{DRIVE_PATH}/flights_2018_2024_v2.parquet', index=False)
print(f'💾 Saved to {DRIVE_PATH}/flights_2018_2024_v2.parquet')

## Section 2: Weather Data Download & Join

Downloads NOAA ASOS hourly weather per airport/year and joins with BTS at T-2 hours before scheduled departure.

In [ ]:
bts = pd.read_parquet(f'{DRIVE_PATH}/flights_2018_2024_v2.parquet')
print(f'BTS loaded: {bts.shape}')
bts.head()

In [ ]:
# Get all unique airport codes (both origin and destination)
airports = sorted(set(bts['Origin'].unique()) | set(bts['Dest'].unique()))
print(f'Total unique airports: {len(airports)}')
print(airports[:10])  # Preview

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
YEARS        = list(range(2018, 2025))     # 2018, 2019, ... 2024
TIMEZONE     = 'America/New_York'
SAVE_DIR     = os.path.join(DRIVE_PATH, 'weather_cache')
DELAY        = 5
OUTPUT_FILE  = os.path.join(DRIVE_PATH, 'bts_with_weather.parquet')

total_jobs = len(airports) * len(YEARS)
print(f"📋 BTS records:     {len(bts):,}")
print(f"✈️  Unique airports: {len(airports)}")
print(f"📅 Years:           {YEARS[0]}–{YEARS[-1]} ({len(YEARS)} years)")
print(f"📦 Total downloads: {len(airports)} airports × {len(YEARS)} years = {total_jobs}")
print(f"⏱️  Estimated time:  ~{(total_jobs * DELAY) // 60} minutes (first run)")
print()

In [ ]:
# ============================================================
# STEP 1: Download function with retry logic
# ============================================================
def fetch_weather(station, year, max_retries=3):
    url = (
        "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?"
        f"station={station}&"
        "data=tmpf&data=dwpf&data=relh&data=sknt&data=gust&"
        "data=vsby&data=p01i&data=wxcodes&data=feel&"
        f"tz={TIMEZONE}&"
        "format=onlycomma&latlon=no&missing=empty&"
        "trace=0.0001&"
        f"year1={year}&month1=1&day1=1&"
        f"year2={year}&month2=12&day2=31"
    )

    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=60)

            if response.status_code == 429:
                wait_time = (attempt + 1) * 30
                print(f"    ⏳ 429 Rate limited. Waiting {wait_time}s "
                      f"(attempt {attempt + 1}/{max_retries})...")
                time.sleep(wait_time)
                continue

            response.raise_for_status()

            df = pd.read_csv(StringIO(response.text))
            if len(df) > 0:
                return df
            return None

        except requests.exceptions.RequestException as e:
            wait_time = (attempt + 1) * 15
            print(f"    ⚠️ Error: {e}. Retrying in {wait_time}s...")
            time.sleep(wait_time)

    return None

In [ ]:
files = os.listdir('weather_cache') if os.path.exists('weather_cache') else []
print(f"Cache files: {len(files)}")

In [ ]:
# ============================================================
# STEP 2: Download weather — one file per airport per year
# ============================================================
print('=' * 60)
print('DOWNLOADING WEATHER DATA')
print('=' * 60)
os.makedirs(SAVE_DIR, exist_ok=True)
failed_jobs = []
job_num = 0
for year in YEARS:
    print(f'\n--- {year} ---')
    for i, airport in enumerate(airports, 1):
        job_num += 1
        cache_file = os.path.join(SAVE_DIR, f'{airport}_{year}.csv')
        if os.path.exists(cache_file):
            print(f'  [{job_num}/{total_jobs}] 📁 {airport} {year}: cached')
            continue
        print(f'  [{job_num}/{total_jobs}] ⬇️  {airport} {year}...', end=' ')
        df = fetch_weather(airport, year, max_retries=3)
        if df is not None:
            df.to_csv(cache_file, index=False)
            print(f'✅ {len(df):,} records')
        else:
            failed_jobs.append((airport, year))
            print('❌ No data')
        time.sleep(DELAY)
print(f'\n  ✅ Downloaded: {total_jobs - len(failed_jobs)}/{total_jobs}')
print()


In [ ]:
# ============================================================
# STEP 3: Retry failed stations with ICAO prefix
# ============================================================
if failed_jobs:
    print('=' * 60)
    print(f'RETRYING {len(failed_jobs)} FAILED DOWNLOADS')
    print('=' * 60)

    still_failed = []

    for airport, year in failed_jobs:
        cache_file = os.path.join(SAVE_DIR, f'{airport}_{year}.csv')
        print(f'  🔄 Retrying {airport} {year} as K{airport}...', end=' ')
        time.sleep(10)

        df = fetch_weather(f'K{airport}', year, max_retries=3)

        if df is not None:
            df['station'] = airport
            df.to_csv(cache_file, index=False)
            print(f'✅ Recovered ({len(df):,} records)')
        else:
            still_failed.append((airport, year))
            print('❌ Permanently failed')

    if still_failed:
        failed_airports = sorted(set(a for a, y in still_failed))
        print(f'\n  ⛔ No data available for: {failed_airports}')
        print(f'     Total failed: {len(still_failed)}')
    print()
else:
    still_failed = []
    print('No failed downloads 🎉\n')

In [ ]:
# ============================================================
# STEP 4: Combine and clean weather data (from disk, memory-safe)
# ============================================================
print('=' * 60)
print('CLEANING WEATHER DATA')
print('=' * 60)
csv_files = sorted([os.path.join(SAVE_DIR, f) for f in os.listdir(SAVE_DIR) if f.endswith('.csv')])
print(f'  Reading {len(csv_files)} cached files...')
numeric_cols = ['tmpf', 'dwpf', 'relh', 'sknt', 'gust', 'vsby', 'p01i', 'feel']
agg_chunks = []
for i, f in enumerate(csv_files):
    df = pd.read_csv(f)
    df['valid'] = pd.to_datetime(df['valid'], errors='coerce')
    df = df.dropna(subset=['valid'])
    df['date'] = df['valid'].dt.date.astype(str)
    df['hour'] = df['valid'].dt.hour
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    agg = df.groupby(['station', 'date', 'hour']).agg(
        temp_f       = ('tmpf', 'mean'),
        dewpoint_f   = ('dwpf', 'mean'),
        humidity     = ('relh', 'mean'),
        feels_like_f = ('feel', 'mean'),
        wind_kts     = ('sknt', 'mean'),
        gust_kts     = ('gust', 'max'),
        visibility   = ('vsby', 'mean'),
        precip_in    = ('p01i', 'sum'),
        wx_codes     = ('wxcodes', lambda x: '|'.join(x.dropna().unique())),
    ).reset_index()
    agg_chunks.append(agg)
    if (i + 1) % 200 == 0:
        print(f'  Processed {i+1}/{len(csv_files)} files...')
weather_hourly = pd.concat(agg_chunks, ignore_index=True)
del agg_chunks
gc.collect()
print(f'  Hourly records: {len(weather_hourly):,}')
print(f'  Stations: {weather_hourly["station"].nunique()}')
print(f'  Date range: {weather_hourly["date"].min()} → {weather_hourly["date"].max()}')
print()

In [ ]:
# ============================================================
# STEP 5: Engineer weather features
# ============================================================
print("=" * 60)
print("ENGINEERING WEATHER FEATURES")
print("=" * 60)

weather_hourly['is_rain'] = weather_hourly['wx_codes'].str.contains(
    'RA|DZ|TS', case=False, na=False).astype(int)

weather_hourly['is_snow'] = weather_hourly['wx_codes'].str.contains(
    'SN|SG|PL|IC', case=False, na=False).astype(int)

weather_hourly['is_fog'] = weather_hourly['wx_codes'].str.contains(
    'FG|BR|HZ', case=False, na=False).astype(int)

weather_hourly['low_visibility'] = (
    weather_hourly['visibility'] < 3).astype(int)

weather_hourly['high_wind'] = (
    weather_hourly['gust_kts'] > 25).astype(int)

weather_hourly['severe_weather'] = (
    weather_hourly['is_rain']        |
    weather_hourly['is_snow']        |
    weather_hourly['low_visibility'] |
    weather_hourly['high_wind']
).astype(int)

weather_hourly.to_parquet(os.path.join(DRIVE_PATH, 'weather_hourly_all_airports.parquet'), index=False)

print("  ✅ Features: is_rain, is_snow, is_fog, "
      "low_visibility, high_wind, severe_weather")
print(f"  💾 Saved: {DRIVE_PATH}/weather_hourly_all_airports.parquet")
print()


In [ ]:
# ============================================================
# LOAD SAVED WEATHER DATA
# ============================================================
print("Loading saved weather data...")

weather_hourly = pd.read_parquet(os.path.join(DRIVE_PATH, 'weather_hourly_all_airports.parquet'))

print(f"  ✅ Loaded: {len(weather_hourly):,} hourly records")
print(f"  Stations: {weather_hourly['station'].nunique()}")
print(f"  Date range: {weather_hourly['date'].min()} → {weather_hourly['date'].max()}")
print(f"  Columns: {list(weather_hourly.columns)}")
print()

OUTPUT_FILE = 'bts_with_weather.parquet'

In [ ]:
# ============================================================
# STEP 6: Prepare BTS join keys
# ============================================================
print("=" * 60)
print("STEP 6: PREPARING BTS MERGE KEYS")
print("=" * 60)

bts['FlightDate']  = pd.to_datetime(bts['FlightDate'])
bts['date']        = bts['FlightDate'].dt.date.astype(str)
bts['dep_hour']    = (bts['CRSDepTime'] // 100).astype('Int64')
bts['arr_hour']    = (bts['CRSArrTime'] // 100).astype('Int64')

# ── New columns: scheduled times minus 2 hours (clamped to 0–23) ──
bts['dep_hour_minus2'] = ((bts['dep_hour'] - 2) % 24).astype('Int64')
bts['arr_hour_minus2'] = ((bts['arr_hour'] - 2) % 24).astype('Int64')

print(f"  BTS records:  {len(bts):,}")
print(f"  Date range:   {bts['FlightDate'].min()} → {bts['FlightDate'].max()}")
print(f"  dep_hour sample (orig → -2h): "
      f"{bts['dep_hour'].iloc[0]} → {bts['dep_hour_minus2'].iloc[0]}")
print(f"  arr_hour sample (orig → -2h): "
      f"{bts['arr_hour'].iloc[0]} → {bts['arr_hour_minus2'].iloc[0]}")
print()

In [ ]:
# ============================================================
# STEP 7: Merge ORIGIN weather
# ============================================================
print("=" * 60)
print("STEP 7: MERGING ORIGIN WEATHER")
print("=" * 60)

origin_wx = weather_hourly.copy()
origin_wx.columns = ['station', 'date', 'hour'] + \
    [f'origin_{c}' for c in origin_wx.columns[3:]]

bts_merged = bts.merge(
    origin_wx,
    left_on  = ['Origin', 'date', 'dep_hour_minus2'],
    right_on = ['station', 'date', 'hour'],
    how      = 'left'
).drop(columns=['station', 'hour'])

print(f"  Match rate: {bts_merged['origin_temp_f'].notna().mean():.1%}")
print()

In [ ]:
# ============================================================
# STEP 8: Merge DESTINATION weather
# ============================================================
print("=" * 60)
print("STEP 8: MERGING DESTINATION WEATHER")
print("=" * 60)

dest_wx = weather_hourly.copy()
dest_wx.columns = ['station', 'date', 'hour'] + \
    [f'dest_{c}' for c in dest_wx.columns[3:]]

bts_merged = bts_merged.merge(
    dest_wx,
    left_on  = ['Dest', 'date', 'arr_hour_minus2'],
    right_on = ['station', 'date', 'hour'],
    how      = 'left'
).drop(columns=['station', 'hour'])

print(f"  Match rate: {bts_merged['dest_temp_f'].notna().mean():.1%}")
print()

In [ ]:
# ============================================================
# STEP 9: Validate and save
# ============================================================
print("=" * 60)
print("STEP 9: VALIDATION & OUTPUT")
print("=" * 60)

print(f"\n  📐 Final shape: {bts_merged.shape}")

print(f"\n  🌤️  Origin match rate:  "
      f"{bts_merged['origin_temp_f'].notna().mean():.1%}")
print(f"  🌤️  Dest match rate:    "
      f"{bts_merged['dest_temp_f'].notna().mean():.1%}")

weather_cols = [c for c in bts_merged.columns
                if c.startswith('origin_') or c.startswith('dest_')]
print(f"\n  🆕 Weather columns added: {len(weather_cols)}")

if 'WEATHER_DELAY' in bts_merged.columns:
    wx_delay = bts_merged[bts_merged['WEATHER_DELAY'] > 0]
    if len(wx_delay) > 0:
        severe_delayed = wx_delay['origin_severe_weather'].mean()
        severe_overall = bts_merged['origin_severe_weather'].mean()
        print(f"\n  🔍 Sanity check:")
        print(f"      Severe weather rate (delayed flights): {severe_delayed:.2f}")
        print(f"      Severe weather rate (all flights):     {severe_overall:.2f}")
        if severe_delayed > severe_overall:
            print("      ✅ Looks correct!")

bts_merged.to_parquet(OUTPUT_FILE, index=False)

print(f"\n  💾 Saved: {OUTPUT_FILE}")
print(f"     Size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.1f} MB")

print("\n" + "=" * 60)
print("✅ PIPELINE COMPLETE")
print("=" * 60)

## Section 3: Holiday & Weekend Flags

Adds `is_weekend` and `is_holiday` (US federal holidays ±1 day) flags.

In [ ]:
df = pd.read_parquet(os.path.join(DRIVE_PATH, 'bts_with_weather.parquet'))
print(f'Loaded: {df.shape}')

In [ ]:
!pip install holidays --quiet

In [ ]:
# is_weekend: DayOfWeek 6=Saturday, 7=Sunday (DOT convention)
df['is_weekend'] = df['DayOfWeek'].isin([6, 7]).astype(int)
# is_holiday: US federal holidays ±1 day window
us_holidays = holidays.US(years=range(2018, 2025))
holiday_dates = set(us_holidays.keys())
holiday_window = (
    holiday_dates
    | {d + timedelta(days=1) for d in holiday_dates}
    | {d - timedelta(days=1) for d in holiday_dates}
)
df['is_holiday'] = df['FlightDate'].dt.date.isin(holiday_window).astype(int)
print("is_weekend value counts:")
print(df['is_weekend'].value_counts())
print("\nis_holiday value counts:")
print(df['is_holiday'].value_counts())
df[['FlightDate', 'DayOfWeek', 'is_weekend', 'is_holiday']].head(10)

In [ ]:
# Save final output
final_path = os.path.join(DRIVE_PATH, 'bts_with_weather_holiday.parquet')
df.to_parquet(final_path, index=False)
print(f'Saved: {final_path}  Shape: {df.shape}')
# Delete intermediate files
print('\nCleaning up intermediate files...')
cache_dir = os.path.join(DRIVE_PATH, 'weather_cache')
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print('  Deleted weather_cache/')
for fname in ['bts_with_weather.parquet', 'flights_2018_2024_v2.parquet', 'weather_hourly_all_airports.parquet']:
    fpath = os.path.join(DRIVE_PATH, fname)
    if os.path.exists(fpath):
        os.remove(fpath)
        print(f'  Deleted {fname}')
print('\nDone. data/ now contains only: bts_with_weather_holiday.parquet')


## Section 4: Temporal Split

> **DISABLED** — Temporal split and feature engineering are handled in notebook 03.
> Run notebook 03 after this notebook completes.

In [ ]:
# !apt-get update -qq
# !apt-get install -y openjdk-11-jdk-headless -qq
# !pip install pyspark --quiet

In [ ]:
# import os, subprocess
# result = subprocess.run(
#     "java -XshowSettings:property -version 2>&1 | grep 'java.home'",
#     shell=True, capture_output=True, text=True
# )
# os.environ['JAVA_HOME'] = result.stdout.strip().split('=')[-1].strip()
# print('JAVA_HOME:', os.environ['JAVA_HOME'])

In [ ]:
# from pyspark.sql import SparkSession
# 
# spark = (
#     SparkSession.builder
#     .appName('TemporalSplit')
#     .master('local[*]')
#     .config('spark.driver.memory', '16g')
#     .getOrCreate()
# )
# spark.conf.set('spark.sql.parquet.int96RebaseModeInRead', 'CORRECTED')
# spark.conf.set('spark.sql.legacy.parquet.nanosAsLong', 'true')
# spark.sparkContext.setLogLevel('WARN')
# print('Spark ready')

In [ ]:
# df = spark.read.parquet(f'{DRIVE_PATH}/bts_with_weather_holiday.parquet')
# print(f'Loaded: {df.count():,} rows')

In [ ]:
# from pyspark.sql import functions as F
# 
# (
#     df.groupBy("Year")
#       .agg(F.count("*").alias("rows"))
#       .orderBy("Year")
#       .show()
# )

In [ ]:
# # Time-based split boundaries
# TRAIN_YEARS    = [2018, 2019, 2020, 2021, 2022]
# VALIDATE_YEARS = [2023]
# TEST_YEARS     = [2024]
# 
# train_df    = df.filter(F.col("Year").isin(TRAIN_YEARS))
# validate_df = df.filter(F.col("Year").isin(VALIDATE_YEARS))
# test_df     = df.filter(F.col("Year").isin(TEST_YEARS))

In [ ]:
# def summarize(name, d):
#     total   = d.count()
#     pos     = d.filter(F.col("ArrDel15") == 1).count()
#     pos_pct = pos / total * 100 if total else 0
#     print(f"{name:10s} | rows: {total:>11,} | delayed: {pos:>10,} ({pos_pct:.2f}%)")
# 
# summarize("TRAIN",    train_df)
# summarize("VALIDATE", validate_df)
# summarize("TEST",     test_df)

In [ ]:
# import shutil, os, glob
# from google.colab import drive
# drive.mount('/content/drive')
# 
# BASE_LOCAL = "/content/flights_split"
# BASE_DRIVE = f"{DRIVE_PATH}/flights_split"
# 
# splits = {
#     "train_2018_2022":    train_df,
#     "validate_2023": validate_df,
#     "test_2024":     test_df,
# }
# 
# os.makedirs(BASE_LOCAL, exist_ok=True)
# 
# for name, sdf in splits.items():
#     print(f"\n→ Writing {name} ...")
# 
#     tmp_dir = f"{BASE_LOCAL}/{name}_tmp"
#     final_file = f"{BASE_LOCAL}/{name}.parquet"
# 
#     # 1. Spark writes to a temp directory with a single part-file
#     (
#         sdf.coalesce(1)
#            .write.mode("overwrite")
#            .parquet(f"file://{tmp_dir}")
#     )
# 
#     # 2. Find the single part-*.parquet file and rename it
#     part_file = glob.glob(f"{tmp_dir}/part-*.parquet")[0]
#     shutil.move(part_file, final_file)
# 
#     # 3. Remove the now-empty temp dir (and its _SUCCESS, .crc files)
#     shutil.rmtree(tmp_dir)
# 
#     # 4. Report size
#     size_gb = os.path.getsize(final_file) / 1024**3
#     print(f"   ✅ {name}.parquet written locally ({size_gb:.2f} GB)")
# 
# # Bulk-copy all single files to Drive
# print("\n→ Copying to Google Drive ...")
# os.makedirs(BASE_DRIVE, exist_ok=True)
# for name in splits.keys():
#     src = f"{BASE_LOCAL}/{name}.parquet"
#     dst = f"{BASE_DRIVE}/{name}.parquet"
#     shutil.copy2(src, dst)
#     print(f"   ✅ Copied {name}.parquet to Drive")
# 
# print(f"\n✅ All splits saved as single files to {BASE_DRIVE}")